# 02 — Build KNN Model

Notebook 3/4 of the Content-Based pipeline.

**Strategy:**
- Load TF-IDF sparse matrix from Notebook 01
- Fit `NearestNeighbors(algorithm='brute', metric='cosine')` on sparse matrix
- NO full distance matrix (O(N^2) would OOM)
- Build row_index <-> book_id mapping

**Output:** `cb_knn_model.pkl`, `cb_item_index_mapping.parquet`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pyarrow.parquet as pq
import pickle, gc, os, psutil
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

def print_ram(label=''):
    mem = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    tag = f' [{label}]' if label else ''
    print(f'  [RAM{tag}] {mem:.2f} GB')

# ============================================================
# CONFIG
# ============================================================
BASE_DIR   = '/content/drive/My Drive/Thesis/book_recsys'
DATA_DIR   = os.path.join(BASE_DIR, 'data/processed')

BOOKS_PQ   = os.path.join(DATA_DIR, 'cb_books.parquet')
SOUP_PQ    = os.path.join(DATA_DIR, 'cb_soup_texts.parquet')
TFIDF_MAT  = os.path.join(DATA_DIR, 'cb_tfidf_item_vector.npz')
KNN_OUT    = os.path.join(DATA_DIR, 'cb_knn_model.pkl')
MAPPING_OUT = os.path.join(DATA_DIR, 'cb_item_index_mapping.parquet')

print('Config done.')
print_ram()

Config done.
  [RAM] 0.24 GB


## 1. Load TF-IDF Sparse Matrix

In [3]:
print('Loading TF-IDF matrix...')
tfidf_matrix = sp.load_npz(TFIDF_MAT)
print(f'  Shape: {tfidf_matrix.shape}')
print(f'  nnz:   {tfidf_matrix.nnz:,}')
print_ram('after load matrix')

Loading TF-IDF matrix...
  Shape: (2360648, 20000)
  nnz:   169,379,014
  [RAM [after load matrix]] 1.51 GB


## 2. Fit KNN Model

Using `brute` algorithm with cosine metric on the sparse TF-IDF matrix. The brute approach stores a reference to the data (no copy), so RAM usage stays low.

In [4]:
if os.path.exists(KNN_OUT):
    print(f'SKIP: {os.path.basename(KNN_OUT)} already exists. Loading...')
    with open(KNN_OUT, 'rb') as f:
        knn = pickle.load(f)
    print('  KNN model loaded.')
else:
    print('Fitting KNN (brute, cosine)...')
    knn = NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1)
    knn.fit(tfidf_matrix)
    print('  KNN fitted.')

    # Save
    with open(KNN_OUT, 'wb') as f:
        pickle.dump(knn, f, protocol=4)
    sz = os.path.getsize(KNN_OUT) / 1024**2
    print(f'  Saved: {os.path.basename(KNN_OUT)} ({sz:.1f} MB)')

print_ram('after KNN')

Fitting KNN (brute, cosine)...
  KNN fitted.
  Saved: cb_knn_model.pkl (1301.3 MB)
  [RAM [after KNN]] 2.78 GB


## 3. Build Item Index Mapping

Map each row index in the TF-IDF matrix to its `book_id`. The order must match the order books were processed in Notebook 01.

In [5]:
if os.path.exists(MAPPING_OUT):
    print(f'SKIP: {os.path.basename(MAPPING_OUT)} already exists.')
    df_mapping = pd.read_parquet(MAPPING_OUT)
else:
    # book_ids come from soup_texts.parquet (same order as TF-IDF rows)
    print('Building index mapping from cb_soup_texts.parquet...')
    book_ids = pq.read_table(SOUP_PQ, columns=['book_id']).column('book_id').to_pylist()

    df_mapping = pd.DataFrame({
        'row_index': range(len(book_ids)),
        'book_id': book_ids,
    })

    assert len(df_mapping) == tfidf_matrix.shape[0], \
        f'Mismatch: mapping={len(df_mapping)}, matrix={tfidf_matrix.shape[0]}'

    df_mapping.to_parquet(MAPPING_OUT, index=False)
    print(f'  Saved: {os.path.basename(MAPPING_OUT)} ({len(df_mapping):,} items)')
    del book_ids
    gc.collect()

# Build lookup dicts
book_id_to_idx = dict(zip(df_mapping['book_id'], df_mapping['row_index']))
idx_to_book_id = dict(zip(df_mapping['row_index'], df_mapping['book_id']))
print(f'  Mapping size: {len(book_id_to_idx):,}')
print_ram('after mapping')

Building index mapping from cb_soup_texts.parquet...
  Saved: cb_item_index_mapping.parquet (2,360,648 items)
  Mapping size: 2,360,648
  [RAM [after mapping]] 3.41 GB


## 4. Demo Inference — Top 10 Similar Books

In [6]:
# Load book metadata for display
print('Loading book metadata for display...')
df_books = pd.read_parquet(BOOKS_PQ, columns=['book_id', 'title', 'author', 'genres', 'average_rating'])
print(f'  Books loaded: {len(df_books):,}')

def get_recommendations(book_id, n=10):
    """Get top-n similar books for a given book_id."""
    idx = book_id_to_idx.get(str(book_id))
    if idx is None:
        print(f'Book not found: {book_id}')
        return None

    query_vec = tfidf_matrix[idx]
    distances, indices = knn.kneighbors(query_vec, n_neighbors=n + 1)

    rec_idx  = indices[0][1:]   # skip self
    rec_sims = 1 - distances[0][1:]  # cosine similarity

    rec_book_ids = [idx_to_book_id[i] for i in rec_idx]
    result = df_books[df_books['book_id'].isin(rec_book_ids)].copy()

    # Preserve order
    id_order = {bid: rank for rank, bid in enumerate(rec_book_ids)}
    result['_rank'] = result['book_id'].map(id_order)
    result['similarity'] = result['book_id'].map(dict(zip(rec_book_ids, np.round(rec_sims, 4))))
    result = result.sort_values('_rank').drop(columns=['_rank']).reset_index(drop=True)
    return result

print('get_recommendations() ready.')

Loading book metadata for display...
  Books loaded: 2,360,648
get_recommendations() ready.


In [7]:
# Demo with a few sample books
DEMO_QUERIES = ['1', '5333265', '1333909']  # First few book_ids

for bid in DEMO_QUERIES:
    meta = df_books[df_books['book_id'] == bid]
    if meta.empty:
        print(f'\nBook {bid} not found in catalog.\n')
        continue

    row = meta.iloc[0]
    print(f'\n{"="*60}')
    print(f'Query: {row["title"]}  |  {row["author"]}  |  {row["genres"]}')
    print(f'{"="*60}')

    recs = get_recommendations(bid, n=10)
    if recs is not None:
        print(recs[['title', 'author', 'genres', 'similarity']].to_string(index=True))

print_ram('after demo')


Query: Harry Potter and the Half-Blood Prince (Harry Potter, #6)  |  J.K. Rowling  |  fantasy, paranormal, young-adult, fiction, children, mystery, thriller, crime, romance
                                                       title        author                                                                                  genres  similarity
0  Harry Potter and the Half-Blood Prince (Harry Potter, #6)  J.K. Rowling  fantasy, paranormal, young-adult, fiction, children, mystery, thriller, crime, romance      0.9912
1  Harry Potter and the Half-Blood Prince (Harry Potter, #6)  J.K. Rowling  fantasy, paranormal, young-adult, fiction, children, mystery, thriller, crime, romance      0.9106
2  Harry Potter and the Half-Blood Prince (Harry Potter, #6)  J.K. Rowling  fantasy, paranormal, young-adult, fiction, children, mystery, thriller, crime, romance      0.9106
3                     Harry Potter and the Half-Blood Prince  J.K. Rowling  fantasy, paranormal, young-adult, fiction, childre

## 5. Summary

In [8]:
print('=' * 55)
print('  KNN MODEL SUMMARY')
print('=' * 55)
print(f'  TF-IDF matrix: {tfidf_matrix.shape}')
print(f'  KNN algorithm: brute (cosine)')
print(f'  Items mapped:  {len(book_id_to_idx):,}')
for name, path in [('cb_knn_model.pkl', KNN_OUT), ('cb_item_index_mapping.parquet', MAPPING_OUT)]:
    if os.path.exists(path):
        sz = os.path.getsize(path) / 1024**2
        print(f'  {name:<40} {sz:>8.1f} MB')
print('=' * 55)
print_ram('final')

  KNN MODEL SUMMARY
  TF-IDF matrix: (2360648, 20000)
  KNN algorithm: brute (cosine)
  Items mapped:  2,360,648
  cb_knn_model.pkl                           1301.3 MB
  cb_item_index_mapping.parquet                25.8 MB
  [RAM [final]] 4.33 GB
